# 03 — OOF Nelder-Mead QWK Threshold Search, Conformal Calibration & Weight Export
Loads the validation out-of-fold predictions (`stage2_val_oof.npz`) saved during Stage-2 training:
1. Optimizes the 4 ordinal decision thresholds `[tau_1, tau_2, tau_3, tau_4]` via `NelderMeadQWKOptimizer`.
2. Calibrates probability temperature $T$ via `TemperatureScaler` (`ECE < 0.04`).
3. Calibrates 95% marginal coverage (`alpha = 0.05`) via `SplitConformalPredictor`.
4. Exports `calibration_params.json` and triggers direct browser download of the trained checkpoints.

In [ ]:
# 1. Mount Google Drive & Load Stage-2 Validation OOF Predictions
import os, sys, json
from pathlib import Path
import subprocess
import numpy as np

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = Path('/content/Retinopathy')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', 'https://github.com/dprincepiriyan/Retinopathy.git', str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    DRIVE_CKPT_DIR = Path('/content/drive/MyDrive/Retinopathy_Checkpoints')
else:
    REPO_DIR = Path.cwd().resolve()
    if REPO_DIR.name == 'colab':
        REPO_DIR = REPO_DIR.parent
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    DRIVE_CKPT_DIR = REPO_DIR / 'checkpoints'

LOCAL_CKPT_DIR = REPO_DIR / 'checkpoints'
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

oof_path = DRIVE_CKPT_DIR / 'stage2_val_oof.npz'
if oof_path.exists():
    data = np.load(oof_path)
    oof_probs = data['probs']
    oof_cont_scores = data['continuous_scores']
    oof_y_true = data['targets']
    print(f'Loaded real Stage-2 OOF validation predictions ({len(oof_y_true)} scans) from {oof_path}')
else:
    print('WARNING: stage2_val_oof.npz not found; running synthetic demonstration calibration...')
    rng = np.random.default_rng(42)
    oof_y_true = rng.choice([0, 1, 2, 3, 4], size=500, p=[0.52, 0.10, 0.25, 0.05, 0.08])
    oof_cont_scores = np.clip(oof_y_true + rng.normal(0.0, 0.35, size=500), 0.0, 4.5)
    logits = -np.abs(oof_cont_scores[:, None] - np.arange(5)[None, :]) * 2.2
    exp_l = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    oof_probs = exp_l / np.sum(exp_l, axis=1, keepdims=True)

In [ ]:
# 2. Optimize Nelder-Mead QWK Thresholds, Temperature Scaling & 95% Split Conformal Predictor
from src.tier4_ensemble import NelderMeadQWKOptimizer, SplitConformalPredictor, TemperatureScaler
from src.utils.metrics import quadratic_weighted_kappa, expected_calibration_error

# A. Nelder-Mead Simplex QWK Threshold Optimization
default_preds = NelderMeadQWKOptimizer.discretize(oof_cont_scores, (0.5, 1.5, 2.5, 3.5))
default_qwk = quadratic_weighted_kappa(oof_y_true, default_preds, num_classes=5)

qwk_opt = NelderMeadQWKOptimizer().fit(oof_cont_scores, oof_y_true)
print(f'Default Thresholds [0.5, 1.5, 2.5, 3.5] QWK : {default_qwk:.4f}')
print(f'Optimized Thresholds {[round(float(t), 4) for t in qwk_opt.thresholds_]} QWK : {qwk_opt.best_qwk_:.4f}')

# B. Temperature Scaling on Pseudo-Logits
pseudo_logits = np.log(np.clip(oof_probs, 1e-8, 1.0))
ece_before = expected_calibration_error(oof_probs, oof_y_true)
temp_scaler = TemperatureScaler().fit_multiclass(pseudo_logits, oof_y_true)
cal_probs = temp_scaler.transform_multiclass(pseudo_logits)
ece_after = expected_calibration_error(cal_probs, oof_y_true)
print(f'Temperature Scaling (T = {temp_scaler.temperature:.4f}): ECE {ece_before:.4f} -> {ece_after:.4f}')

# C. Split Conformal Calibration (95% Marginal Coverage Guarantee)
conformal = SplitConformalPredictor(alpha=0.05).fit(cal_probs, oof_y_true)
covered = 0
for p_row, y_t in zip(cal_probs, oof_y_true):
    res = conformal.predict_single(p_row)
    if int(y_t) in res.prediction_set:
        covered += 1
empirical_cov = covered / max(len(oof_y_true), 1)
print(f'Conformal Non-Conformity Quantile (q_hat): {conformal.q_hat_:.4f} | Empirical Coverage: {empirical_cov * 100:.2f}%')

In [ ]:
# 3. Export Calibration Parameters & Download Checkpoints to Local Laptop
calibration_payload = {
    'qwk_thresholds': [round(float(t), 6) for t in qwk_opt.thresholds_],
    'best_oof_qwk': round(float(qwk_opt.best_qwk_), 4),
    'temperature': round(float(temp_scaler.temperature), 6),
    'conformal_alpha': 0.05,
    'conformal_q_hat': round(float(conformal.q_hat_), 6),
    'empirical_coverage': round(float(empirical_cov), 4),
}

for target_dir in {DRIVE_CKPT_DIR, LOCAL_CKPT_DIR}:
    target_dir.mkdir(parents=True, exist_ok=True)
    with open(target_dir / 'calibration_params.json', 'w', encoding='utf-8') as f:
        json.dump(calibration_payload, f, indent=2)

print('Saved calibration_params.json:', json.dumps(calibration_payload, indent=2))

if IN_COLAB:
    from google.colab import files
    for fname in ['calibration_params.json', 'stage1_multi_label_best.pth', 'stage2_resvit_pro_768_best.pth']:
        fpath = DRIVE_CKPT_DIR / fname
        if fpath.exists():
            print(f'Triggering browser download for {fname} ...')
            files.download(str(fpath))
        else:
            print(f'Note: {fname} not found in {DRIVE_CKPT_DIR} yet.')